# Project 2

셀을 차례대로 실행하면 됩니다.

In [2]:
'''
1. Clone or Pull Repository
코드는 github에 올려서 사용하였습니다.
'''

%cd /content/

import os
REPO_URL = "https://github.com/cyunchaeskku/AILab_Project2_submission.git"
REPO_NAME = "AILab_Project2_submission"

if os.path.exists(REPO_NAME):
    print(f"Pulling latest changes for {REPO_NAME}...")
    !cd {REPO_NAME} && git pull
else:
    print(f"Cloning {REPO_NAME}...")
    !git clone {REPO_URL}

/content
Pulling latest changes for AILab_Project2_submission...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 9 (delta 3), reused 3 (delta 3), pack-reused 6 (from 1)
Unpacking objects: 100% (9/9), 75.15 MiB | 8.34 MiB/s, done.
From https://github.com/cyunchaeskku/AILab_Project2_submission
   62f14b4..1d4c550  main       -> origin/main
Updating 62f14b4..1d4c550
Fast-forward
 README.md                |   4 +
 ckpt/model.pt            |   3 +
 generate_512_upsample.py |  96 +++++
 main.ipynb               |  46 ++-
 pyproject.toml           |  18 +
 submission.onnx          | Bin 0 -> 85120842 bytes
 test.ipynb               | 310 ----------------
 train_512.ipynb          | 932 -----------------------------------------------
 8 files changed, 148 insertions(+), 1261 deletions(-)
 create mode 100644 ckpt/model.pt
 create mode 100644 generate_512_upsample.py
 create mode 100644 pyproject.toml
 create mode 100644 submission.onnx
 delete mod

In [3]:
'''
2. Install dependencies
requirements.txt를 설치합니다.
'''
!pip install gdown
%cd AILab_Project2_submission
!pip install -r requirements.txt

/content/AILab_Project2_submission
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 19.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 5.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 22.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 133.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 131.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.3/338.3 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 147.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 28.5 MB/s eta 0:00:00
  Created wheel for basicsr: filename=basicsr-1.4.2-py3-none-a

In [ ]:
'''
3. Mount Google Drive for Checkpoints
google drive를 mount합니다.
체크포인트를 google drive에 저장하기 위해서 mount는 필수입니다.
'''
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/OpenAILab/project2")
!ls -lh /content/drive/MyDrive/OpenAILab/project2

Mounted at /content/drive
total 48K
drwx------ 2 root root 4.0K May 28 07:46 1024
drwx------ 2 root root 4.0K May 27 15:44 512
drwx------ 2 root root 4.0K May 27 15:35 checkpoints
drwx------ 2 root root 4.0K May 28 14:00 pggan_1024
drwx------ 2 root root 4.0K May 31 08:55 pggan_1024_fp32_safer
drwx------ 2 root root 4.0K May 30 12:38 pggan_1024_fp32_stable
drwx------ 2 root root 4.0K May 31 16:41 pggan_1024_v2
drwx------ 2 root root 4.0K Jun  4 17:18 pggan_1024_v3
drwx------ 2 root root 4.0K Jun  6 03:43 pggan_512_v4
drwx------ 2 root root 4.0K Jun  8 08:07 pggan_512_v4_fine_tune
drwx------ 2 root root 4.0K Jun  5 14:22 pggan_512_wide
drwx------ 2 root root 4.0K May 31 02:53 train_val_data


In [ ]:
'''
4. Download Train/Validation Datasets
데이터셋을 다운로드 합니다.
'''
from pathlib import Path
import shutil
import subprocess

DATA_DIR = Path("data")
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/OpenAILab/project2/train_val_data")
DATA_DIR.mkdir(exist_ok=True)
DRIVE_DATA_DIR.mkdir(parents=True, exist_ok=True)

files = {
    # Training data only. Use these for model training.
    "train_50k_256.zip": "1sz8s31qCWwuCTWT_DQK2_yoQ6pjtUZvG",
    "train_50k_512.zip": "1R1JBbYGfeOPZedKHjuJ5VZavVdjXuJl-",
    "train_50k_1024.zip": "1qFJN3QOmM8kjwgWKztHPIqTvZFuZnZLd",

    # Validation data only. Do not use these for training.
    "valid_10k_256.zip": "1yEA26_FEgHwPur2jZCr0lmONesA6TyI2",
    "valid_10k_512.zip": "1hLDAyFgTZA2KRYSdJvPLhHJ3K0AgWfpE",
    "valid_10k_1024.zip": "1V0HfqDxUytt-wOiqyerXYC1-61EE8HqZ",
}

for filename, file_id in files.items():
    out = DATA_DIR / filename
    cached = DRIVE_DATA_DIR / filename

    if out.exists() and out.stat().st_size > 0:
        print(f"skip local {out} ({out.stat().st_size / 1024**3:.2f} GB)")
        if not cached.exists() or cached.stat().st_size == 0:
            print(f"cache to Drive {cached}")
            shutil.copy2(out, cached)
        continue

    if cached.exists() and cached.stat().st_size > 0:
        print(f"copy from Drive cache {cached} -> {out}")
        shutil.copy2(cached, out)
        continue

    print(f"download {out}")
    url = f"https://drive.google.com/uc?id={file_id}"
    result = subprocess.run(["gdown", url, "-O", str(out)], check=False)
    if result.returncode != 0 or not out.exists() or out.stat().st_size == 0:
        print(f"download failed: {filename}")
        if out.exists() and out.stat().st_size == 0:
            out.unlink()
        continue

    print(f"cache to Drive {cached}")
    shutil.copy2(out, cached)

!ls -lh data
!ls -lh /content/drive/MyDrive/OpenAILab/project2/train_val_data


In [ ]:
'''
5. Verify Baseline Sample
baseline checkpoint로부터 샘플을 생성하여 체크할 수 있습니다.
'''
!python generate.py --ckpt ckpt/ffhq256_baseline.pt --out sample_256.png --n 16 --seed 42

In [5]:
'''
6. Login to Weights & Biases
wandb에 로그인합니다. (api key 직접 입력 필요)
'''
import getpass
import os
import wandb

if not os.environ.get("WANDB_API_KEY"):
    os.environ["WANDB_API_KEY"] = getpass.getpass("WANDB_API_KEY: ")

wandb.login(key=os.environ["WANDB_API_KEY"])


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cyunchaeskku (cyunchaeskku-sungkyunkwan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
'''
7. Clean GPU Processes
OOM 에러를 방지하기 위해 혹시나 실행되고 있는 다른 GPU 프로세스를 종료합니다.
'''
import gc
import os
import signal
import subprocess

print("GPU processes before cleanup:")
!nvidia-smi

current_pid = os.getpid()
cmd = [
    "nvidia-smi",
    "--query-compute-apps=pid,process_name,used_memory",
    "--format=csv,noheader,nounits",
]
result = subprocess.run(cmd, capture_output=True, text=True)

for line in result.stdout.strip().splitlines():
    if not line.strip():
        continue
    pid_text, name, mem_text = [part.strip() for part in line.split(",", 2)]
    pid = int(pid_text)
    if pid == current_pid:
        print(f"keep current kernel pid={pid} mem={mem_text}MiB")
        continue
    print(f"kill gpu process pid={pid} name={name} mem={mem_text}MiB")
    os.kill(pid, signal.SIGTERM)

gc.collect()
try:
    import torch
    torch.cuda.empty|_cache()
except Exception as exc:
    print(f"torch cache cleanup skipped: {exc}")

print("GPU processes after cleanup:")
!nvidia-smi


In [ ]:
'''
8. Start True Progressive Training from Professor 256 Baseline
**학습에는 A100을 사용하였습니다**
--auto-resume을 사용해 체크포인트 파일 중 가장 최근의 체크포인트부터 자동으로 resume합니다.
첫 실행이 아닌 경우 --new-wandb-run은 제거하거나 주석처리합니다.
'''
!python train.py \
  --config configs/progressive_1024_a100.yaml \
  --init-from /ckpt/ffhq256_baseline.pt \ 
  --auto-resume \
  --new-wandb-run

In [7]:
'''
9. Generate Images
seed_number를 바꾸어 이미지를 생성할 수 있습니다.
'''

seed_number = 42

!python generate_512_upsample.py \
    --ckpt ckpt/model.pt \
    --psi 0.75 \
    --out samples/sample_{seed_number}.png --n 24 --seed {seed_number}

Loaded G_ema (21.37M params, z_dim=512)
Truncation: psi=0.75
Progressive generation: resolution=512, alpha=1.000
Native res: (512, 512) -> upsampled to (1024, 1024)
Saved 24 samples -> samples/sample_512_before_1024_bilinear_42.png


In [ ]:
'''
10. Export Progressive Generator to ONNX
저장된 체크포인트를 .onnx로 만듭니다. (체크포인트를 model.pt로 저장하여 ckpt/에 넣어두었음)
기존 export_onnx.py를 일부 수정해 truncation을 추가한 export_onnx_truncation.py를 만들었습니다.
'''
!python export_onnx_truncated.py \
--psi 0.75 \
--ckpt ckpt/model.pt \
--out submission.onnx